# 03 — Preprocessing and Baseline Model

Builds the shared preprocessing pipeline (`src/preprocessing.py`) and a logistic-regression
baseline, then reproduces the project's central lesson: a model trained with the leakage feature
`duration` looks dramatically better while being unusable in production.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'src').exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from src.data_loader import load_bank_data
from src.preprocessing import split_features_target
from src.modeling import build_pipeline
from src.model_comparison import compute_metrics
from src.config import RANDOM_STATE, TEST_SIZE

raw_df = load_bank_data()
print('Raw shape:', raw_df.shape)

Raw shape: (41188, 21)


## Realistic baseline (no `duration`)

In [2]:
X, y = split_features_target(raw_df)  # drop_leakage=True by default: duration is removed here
print('Feature columns:', list(X.columns))
print('Duplicate rows dropped:', len(raw_df) - len(X))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)

baseline = build_pipeline(LogisticRegression(max_iter=3000), X_train)
baseline.fit(X_train, y_train)
baseline_probs = baseline.predict_proba(X_test)[:, 1]
baseline_metrics = compute_metrics(y_test, baseline_probs)
for name, value in baseline_metrics.items():
    print(f'  {name:20} {value:.4f}')

Feature columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Duplicate rows dropped: 12


  accuracy             0.8983
  precision            0.6500
  recall               0.2101
  f1                   0.3176
  roc_auc              0.8004
  average_precision    0.4541


## Leakage benchmark (`duration` included)

Same split, same preprocessing shape, the only difference is one column.

In [3]:
X_leak, y_leak = split_features_target(raw_df, drop_leakage=False)  # keeps duration
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leak, y_leak, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_leak,
)

leakage_model = build_pipeline(LogisticRegression(max_iter=3000), Xl_train)
leakage_model.fit(Xl_train, yl_train)
leakage_probs = leakage_model.predict_proba(Xl_test)[:, 1]
leakage_metrics = compute_metrics(yl_test, leakage_probs)
for name, value in leakage_metrics.items():
    print(f'  {name:20} {value:.4f}')

  accuracy             0.9095
  precision            0.6522
  recall               0.4224
  f1                   0.5128
  roc_auc              0.9390
  average_precision    0.6014


In [4]:
comparison = pd.DataFrame({'realistic (no duration)': baseline_metrics, 'leakage (+ duration)': leakage_metrics})
comparison.round(4)

,realistic (no duration),leakage (+ duration)
accuracy,0.8983,0.9095
precision,0.6500,0.6522
recall,0.2101,0.4224
f1,0.3176,0.5128
roc_auc,0.8004,0.9390
average_precision,0.4541,0.6014


The leakage model's ROC-AUC and recall are dramatically higher — it has, in effect, learned 'long calls convert,' which is true but useless: nobody knows how long a call will last before making it. Every model compared from here on excludes `duration`; see notebook 04 for the systematic comparison across model types and notebook 01 for the leakage mechanism.